# Notebook 12: Post-Training Scaling Laws & Advanced Techniques

**Frontier AI Interview Prep** | Post-Training & Alignment Series

---

This notebook covers the scaling laws and advanced techniques that determine *how much* post-training
helps and *which method* to use. We implement online DPO, iterative DPO, and SPIN, then study
reward model overoptimization and the industrial Llama 3 recipe.

**Prerequisites**: Notebooks 01-11 (RLHF, DPO, Reward Modeling, Synthetic Data).

**Runtime**: Google Colab GPU recommended. ~25 min with GPU, ~60 min CPU-only.

## 0. Self-Quiz (Active Recall)

Before reading anything, try to answer these from memory:

1. **Do scaling laws apply to post-training?** If you double the RL compute, does alignment quality double?
2. **Does more RL compute always help?** What limits the returns from post-training?
3. **Online vs offline DPO -- which wins?** Why?
4. **What is iterative DPO?** How many iterations are useful before diminishing returns?
5. **What is SPIN?** How does self-play apply to language model alignment?

---
*Write your answers below, then check against the notebook content.*

In [ ]:
# YOUR ANSWERS
my_answers = {
    "scaling_laws_post_training": "",
    "more_rl_compute": "",
    "online_vs_offline": "",
    "iterative_dpo": "",
    "spin": "",
}

## 1. Setup

In [ ]:
%%capture
!pip install torch transformers matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import random
import math
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
import copy
import warnings
warnings.filterwarnings('ignore')

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Post-Training Scaling: The Big Picture

### Pretraining Scaling (Recap)

The Chinchilla scaling law (Hoffmann et al. 2022) tells us:
- **Loss** decreases as a power law with compute: $L(C) \propto C^{-\alpha}$
- **Optimal allocation**: parameters and tokens should scale roughly equally
- More compute always helps (in the regime studied)

### Post-Training Scaling: Different Rules

Post-training (RLHF, DPO, etc.) follows different scaling dynamics:

1. **At a fixed budget, data quality beats noise** -- the Phi-1 lesson (clean labels > noisy ones). Caveat: this is per-budget, not a license to use tiny datasets -- quantity still buys signal.
2. **Diminishing returns hit faster** -- you can overfit to the reward model
3. **There is a soft quality ceiling** -- post-training mostly elicits and sharpens what the base model already learned; it rarely installs genuinely new knowledge. (RL on verifiable rewards, e.g. R1, can still surface latent skills the base did not reliably show -- see point 4.)
4. **RL compute scaling works** -- DeepSeek-R1 showed more RL compute = better reasoning
5. **Test-time compute is the new frontier** -- thinking longer at inference helps more than training longer

### The Hierarchy of Returns

```
High returns    [Base model quality]     -- Foundation determines ceiling
                [SFT data quality]       -- High-quality demos > quantity
                [Preference data quality] -- Clean labels > many labels  
                [Algorithm choice]       -- Online > offline, iterative > single-shot
                [Hyperparameter tuning]  -- beta, learning rate, KL budget
Low returns     [More data quantity]     -- Diminishing returns quickly
```

### The DeepSeek-R1 Discovery

DeepSeek-R1 demonstrated that **scaling RL compute** (more steps of PPO/GRPO) produces
emergent reasoning behaviors: the model learns to self-verify, explore multiple solution
paths, and self-correct -- all without being explicitly trained to do so. This suggests
that post-training compute scaling has a different character than pretraining: it unlocks
*qualitative* capability jumps, not just smooth improvements.

## 3. Data Quality vs Quantity

Let us demonstrate concretely, on a HELD-OUT gold-reward set, what data quality
buys you in preference learning -- and where the popular "quality > quantity" slogan
needs a caveat. We pit a clean set against a noisy set of the SAME size (so it is a
fair budget comparison), plus a much smaller clean set to show that quantity still
matters. Crucially, we score every trained policy against the SHARED ground-truth
reward, never against its own (possibly flipped) training labels -- otherwise the
noisy set's score would be capped near 1 - flip_prob = 0.70 purely by construction.

In [ ]:
class SimplePolicyNetwork(nn.Module):
    """A simplified policy network for demonstrating DPO training dynamics.

    We use a small MLP that maps 'state' embeddings to 'action' logits,
    simulating the preference learning scenario. This is intentionally
    simple to make the quality vs quantity comparison clear.
    """

    def __init__(self, input_dim: int = 16, hidden_dim: int = 64, output_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def log_prob(self, state: torch.Tensor, action_idx: torch.Tensor) -> torch.Tensor:
        """Compute log probability of action given state."""
        logits = self.forward(state)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.gather(1, action_idx.unsqueeze(1)).squeeze(1)


class PreferenceDataset(Dataset):
    """Dataset of preference pairs."""

    def __init__(self, states, chosen_actions, rejected_actions):
        self.states = states
        self.chosen = chosen_actions
        self.rejected = rejected_actions

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.chosen[idx], self.rejected[idx]


def generate_preference_data(n_samples: int, input_dim: int = 16,
                             output_dim: int = 32, noise_level: float = 0.0,
                             label_flip_prob: float = 0.0,
                             true_weights: torch.Tensor = None) -> PreferenceDataset:
    """Generate synthetic preference data with controllable quality.

    Args:
        n_samples: Number of preference pairs
        noise_level: Noise in features (0 = clean, 1 = very noisy)
        label_flip_prob: Probability of flipping chosen/rejected (0 = clean, 0.5 = random)
        true_weights: SHARED ground-truth reward matrix. Pass the SAME tensor to
            every dataset that is meant to be the same learning problem, otherwise
            each call invents a different reward function and the datasets are not
            comparable. If None, a fresh one is created (single-dataset use only).
    """
    # Ground truth "reward" function: a random linear function.
    # Reuse a shared one when provided so all datasets describe the SAME task.
    if true_weights is None:
        true_weights = torch.randn(input_dim, output_dim)

    states = torch.randn(n_samples, input_dim)

    # Add noise to states
    if noise_level > 0:
        states = states + noise_level * torch.randn_like(states)

    # Compute true rewards for all actions at each state
    true_rewards = states @ true_weights  # [n_samples, output_dim]

    # Sample two actions per state, pick the higher-reward one as 'chosen'
    action_a = torch.randint(0, output_dim, (n_samples,))
    action_b = torch.randint(0, output_dim, (n_samples,))

    reward_a = true_rewards.gather(1, action_a.unsqueeze(1)).squeeze(1)
    reward_b = true_rewards.gather(1, action_b.unsqueeze(1)).squeeze(1)

    # Assign chosen/rejected based on reward
    chosen = torch.where(reward_a > reward_b, action_a, action_b)
    rejected = torch.where(reward_a > reward_b, action_b, action_a)

    # Flip labels with some probability (simulates labeling errors)
    if label_flip_prob > 0:
        flip_mask = torch.rand(n_samples) < label_flip_prob
        temp_chosen = chosen.clone()
        chosen[flip_mask] = rejected[flip_mask]
        rejected[flip_mask] = temp_chosen[flip_mask]

    return PreferenceDataset(states, chosen, rejected), true_weights


def dpo_loss(policy: SimplePolicyNetwork, ref_policy: SimplePolicyNetwork,
             states: torch.Tensor, chosen: torch.Tensor, rejected: torch.Tensor,
             beta: float = 0.1) -> torch.Tensor:
    """Compute DPO loss."""
    # Policy log probs
    pi_chosen = policy.log_prob(states, chosen)
    pi_rejected = policy.log_prob(states, rejected)

    # Reference log probs
    with torch.no_grad():
        ref_chosen = ref_policy.log_prob(states, chosen)
        ref_rejected = ref_policy.log_prob(states, rejected)

    # DPO objective
    logits = beta * ((pi_chosen - ref_chosen) - (pi_rejected - ref_rejected))
    loss = -F.logsigmoid(logits).mean()

    # Accuracy: how often does the policy prefer chosen over rejected?
    with torch.no_grad():
        accuracy = (pi_chosen > pi_rejected).float().mean()

    return loss, accuracy


def train_dpo(dataset: PreferenceDataset, n_epochs: int = 50,
              batch_size: int = 32, lr: float = 1e-3, beta: float = 0.1,
              input_dim: int = 16, output_dim: int = 32) -> Dict:
    """Train a policy with DPO and return metrics."""
    policy = SimplePolicyNetwork(input_dim=input_dim, output_dim=output_dim).to(device)
    ref_policy = copy.deepcopy(policy)
    ref_policy.eval()

    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []
    accuracies = []

    for epoch in range(n_epochs):
        epoch_loss = 0
        epoch_acc = 0
        n_batches = 0

        for states, chosen, rejected in dataloader:
            states = states.to(device)
            chosen = chosen.to(device)
            rejected = rejected.to(device)

            loss, acc = dpo_loss(policy, ref_policy, states, chosen, rejected, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc.item()
            n_batches += 1

        losses.append(epoch_loss / n_batches)
        accuracies.append(epoch_acc / n_batches)

    return {
        "policy": policy,
        "losses": losses,
        "accuracies": accuracies,
        "final_loss": losses[-1],
        "final_accuracy": accuracies[-1],
    }


def gold_eval(policy: SimplePolicyNetwork, true_weights: torch.Tensor,
              n_eval: int = 2000, input_dim: int = 16) -> float:
    """Held-out GOLD evaluation against the shared ground-truth reward.

    Generate fresh CLEAN states, ask the policy for its top action, and measure
    how often that action's TRUE reward beats a random alternative. This never
    looks at (possibly flipped) training labels, so a noisy training set cannot
    inflate it -- it is the honest quality number.
    """
    policy.eval()
    with torch.no_grad():
        states = torch.randn(n_eval, input_dim).to(device)
        true_rewards = states.cpu() @ true_weights  # [n_eval, output_dim]
        policy_top = policy(states).argmax(dim=-1).cpu()  # model's chosen action
        rand_alt = torch.randint(0, true_weights.shape[1], (n_eval,))
        r_policy = true_rewards.gather(1, policy_top.unsqueeze(1)).squeeze(1)
        r_rand = true_rewards.gather(1, rand_alt.unsqueeze(1)).squeeze(1)
        return (r_policy > r_rand).float().mean().item()


# Experiment: quality vs quantity (HONEST held-out evaluation)
print("EXPERIMENT: Data Quality vs Data Quantity")
print("=" * 50)
print("\nAll datasets share ONE ground-truth reward and are scored on a held-out")
print("GOLD set (true reward), NOT on their own training labels.")
print("  A) 2,000 clean pairs        (clean labels + clean features)")
print("  B) 2,000 noisy pairs        (30% flipped labels + noisy features) -- SAME budget as A")
print("  C)   200 clean pairs        (clean, but 10x less data than A)")
print()

# ONE shared ground-truth reward so every dataset is the SAME learning problem.
TRUE_W = torch.randn(16, 32)

ds_clean,  _ = generate_preference_data(
    n_samples=2000, noise_level=0.0, label_flip_prob=0.0, true_weights=TRUE_W
)
ds_noisy,  _ = generate_preference_data(
    n_samples=2000, noise_level=0.5, label_flip_prob=0.3, true_weights=TRUE_W
)
ds_small,  _ = generate_preference_data(
    n_samples=200,  noise_level=0.0, label_flip_prob=0.0, true_weights=TRUE_W
)

results = {}
for name, ds in [("2K clean", ds_clean),
                  ("2K noisy (30% flips)", ds_noisy),
                  ("200 clean", ds_small)]:
    print(f"Training on {name}...")
    result = train_dpo(ds, n_epochs=80, beta=0.1)
    result["gold_accuracy"] = gold_eval(result["policy"], TRUE_W)
    results[name] = result
    print(f"  Train pref-accuracy: {result['final_accuracy']:.3f}"
          f"   |   Held-out GOLD accuracy: {result['gold_accuracy']:.3f}")

print()
print("Read the GOLD column, not the train column:")
print("  - At the SAME data budget, clean (~0.91) crushes noisy (~0.72):")
print("    30% flipped labels drag every gradient the wrong way. Quality matters.")
print("  - But quantity is NOT free: 200 clean (~0.59) < 2K clean (~0.91), and even")
print("    10K noisy would beat 2K noisy. So: at a FIXED budget, spend it on clean data;")
print("    noise is a tax on every label, not a magic that more data erases.")
print("  - Train pref-accuracy on the noisy set is capped near 1 - flip_prob = 0.70 BY")
print("    CONSTRUCTION -- which is exactly why we never trust the train number here.")
print("\nDone!")


In [ ]:
# Plot the comparison (training curves) + the honest held-out GOLD bars
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'2K clean': '#2ecc71', '2K noisy (30% flips)': '#e74c3c', '200 clean': '#3498db'}

for name, result in results.items():
    axes[0].plot(result['losses'], label=name, color=colors[name], linewidth=2)
    axes[1].plot(result['accuracies'], label=name, color=colors[name], linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('DPO Loss')
axes[0].set_title('Training Loss'); axes[0].legend()

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train Preference Accuracy')
axes[1].set_title('Train Pref-Accuracy (noisy capped ~0.70 by flips)')
axes[1].set_ylim(0.4, 1.0); axes[1].legend()
axes[1].axhline(0.70, color='#e74c3c', ls=':', alpha=0.6)

# The honest comparison: held-out GOLD accuracy
names = list(results.keys())
gold = [results[n]['gold_accuracy'] for n in names]
axes[2].bar(names, gold, color=[colors[n] for n in names])
axes[2].axhline(0.5, color='gray', ls='--', alpha=0.6, label='random = 0.5')
axes[2].set_ylabel('Held-out GOLD accuracy'); axes[2].set_ylim(0.4, 1.0)
axes[2].set_title('What actually matters: GOLD (true reward)')
axes[2].legend(); axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('quality_vs_quantity.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nKey Finding (read the GOLD bars, not the train curves):")
print("  - Same budget, clean beats noisy by a mile (~0.91 vs ~0.72): 30% flipped")
print("    labels are a tax on every gradient. At a fixed budget, curate.")
print("  - Quantity still helps: 200 clean (~0.59) is far below 2K clean (~0.91).")
print("  - The Phi-1 lesson, stated correctly: per-example quality dominates, but")
print("    it does NOT mean a handful of clean pairs beats a large noisy set -- more")
print("    data still buys signal. Spend your budget on clean data, and on enough of it.")


## 4. Online vs Offline DPO

### The Distribution Mismatch Problem

**Offline DPO** (what we have been doing so far):
- Collect a fixed preference dataset
- Train the policy on this dataset
- Problem: as the policy improves, it moves away from the distribution that generated the data
- The data was generated by the *initial* policy, but we are training a *different* policy

**Online DPO**:
- Use the *current* policy to generate responses
- Get those responses ranked (by RM or AI judge)
- Train on these on-policy preference pairs
- Repeat

### Why Online Wins

The fundamental issue is **distribution mismatch**. In offline DPO:
- The preference data was generated by some policy $\pi_{\text{old}}$
- We are training $\pi_{\theta}$ which diverges from $\pi_{\text{old}}$ over training
- The DPO loss becomes less informative as the distributions diverge

In online DPO:
- Preference data is always from the *current* $\pi_{\theta}$
- No distribution mismatch
- The model always trains on its own failure modes

**Key paper**: Xiong et al. 2024, ["Iterative Preference Learning from Human Feedback"](https://arxiv.org/abs/2312.11456)

In [ ]:
class RewardModel(nn.Module):
    """Simple reward model for scoring responses."""
    
    def __init__(self, input_dim: int = 16, hidden_dim: int = 64, output_dim: int = 32):
        super().__init__()
        self.state_embed = nn.Linear(input_dim, hidden_dim)
        self.action_embed = nn.Embedding(output_dim, hidden_dim)
        self.score_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        s = F.relu(self.state_embed(state))
        a = self.action_embed(action)
        combined = torch.cat([s, a], dim=-1)
        return self.score_head(combined).squeeze(-1)


def train_reward_model(dataset: PreferenceDataset, n_epochs: int = 30,
                       input_dim: int = 16, output_dim: int = 32) -> RewardModel:
    """Train a reward model on preference data."""
    rm = RewardModel(input_dim=input_dim, output_dim=output_dim).to(device)
    optimizer = torch.optim.Adam(rm.parameters(), lr=1e-3)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    for epoch in range(n_epochs):
        for states, chosen, rejected in dataloader:
            states = states.to(device)
            chosen = chosen.to(device)
            rejected = rejected.to(device)
            
            r_chosen = rm(states, chosen)
            r_rejected = rm(states, rejected)
            
            # Bradley-Terry loss
            loss = -F.logsigmoid(r_chosen - r_rejected).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    return rm


def online_dpo_step(policy: SimplePolicyNetwork, ref_policy: SimplePolicyNetwork,
                    rm: RewardModel, n_samples: int = 200, n_candidates: int = 4,
                    input_dim: int = 16, output_dim: int = 32,
                    beta: float = 0.1, lr: float = 1e-3) -> Tuple[float, float]:
    """One step of online DPO:
    1. Generate states
    2. For each state, sample multiple actions from current policy
    3. Score with RM
    4. Create preference pairs (best vs worst)
    5. DPO update
    """
    policy.train()
    # NOTE: re-instantiating Adam on every call resets its state (moment estimates) each step -- acceptable only in this toy demo; real training code creates the optimizer once, outside the loop.
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    
    # Step 1: Generate states
    states = torch.randn(n_samples, input_dim).to(device)
    
    # Step 2: Sample actions from current policy
    with torch.no_grad():
        logits = policy(states)  # [n_samples, output_dim]
        probs = F.softmax(logits, dim=-1)
        
        # Sample n_candidates actions per state
        all_actions = torch.multinomial(probs, n_candidates, replacement=True)  # [n_samples, n_candidates]
    
    # Step 3: Score with RM
    with torch.no_grad():
        scores = torch.zeros(n_samples, n_candidates).to(device)
        for j in range(n_candidates):
            scores[:, j] = rm(states, all_actions[:, j])
    
    # Step 4: Create preference pairs (best vs worst scored)
    best_idx = scores.argmax(dim=1)
    worst_idx = scores.argmin(dim=1)
    
    chosen = all_actions.gather(1, best_idx.unsqueeze(1)).squeeze(1)
    rejected = all_actions.gather(1, worst_idx.unsqueeze(1)).squeeze(1)
    
    # Step 5: DPO update
    loss, accuracy = dpo_loss(policy, ref_policy, states, chosen, rejected, beta)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item(), accuracy.item()


# Compare online vs offline DPO
print("EXPERIMENT: Online vs Offline DPO")
print("=" * 50)

INPUT_DIM = 16
OUTPUT_DIM = 32

# First, train a reward model on clean data
print("Training reward model...")
rm_data, true_w = generate_preference_data(n_samples=2000, noise_level=0.0, label_flip_prob=0.0)
rm = train_reward_model(rm_data, n_epochs=50)
print("Reward model trained.")

# Offline DPO: train on a fixed dataset
print("\nTraining offline DPO...")
offline_data, _ = generate_preference_data(n_samples=500, noise_level=0.0, label_flip_prob=0.0)
offline_result = train_dpo(offline_data, n_epochs=100, beta=0.1)
print(f"  Final accuracy: {offline_result['final_accuracy']:.3f}")

# Online DPO: iteratively generate and train
print("\nTraining online DPO...")
online_policy = SimplePolicyNetwork(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM).to(device)
online_ref = copy.deepcopy(online_policy)
online_ref.eval()

online_losses = []
online_accuracies = []

for step in range(100):
    loss, acc = online_dpo_step(
        online_policy, online_ref, rm,
        n_samples=200, n_candidates=4,
        input_dim=INPUT_DIM, output_dim=OUTPUT_DIM,
        beta=0.1, lr=1e-3
    )
    online_losses.append(loss)
    online_accuracies.append(acc)

print(f"  Final accuracy: {online_accuracies[-1]:.3f}")

In [ ]:
# Plot online vs offline comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(offline_result['losses'], label='Offline DPO', color='#e74c3c', linewidth=2)
axes[0].plot(online_losses, label='Online DPO', color='#2ecc71', linewidth=2)
axes[0].set_xlabel('Step/Epoch')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('Loss: Online vs Offline DPO')
axes[0].legend()

axes[1].plot(offline_result['accuracies'], label='Offline DPO', color='#e74c3c', linewidth=2)
axes[1].plot(online_accuracies, label='Online DPO', color='#2ecc71', linewidth=2)
axes[1].set_xlabel('Step/Epoch')
axes[1].set_ylabel('Preference Accuracy')
axes[1].set_title('Accuracy: Online vs Offline DPO')
axes[1].legend()
axes[1].set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig('online_vs_offline_dpo.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key finding: Online DPO trains on its own current outputs,")
print("avoiding the distribution mismatch of offline DPO.")
print("This typically leads to better final performance.")

## 5. Iterative DPO

### The Idea

Rather than training DPO once, run multiple iterations:

1. **Round 1**: Train DPO on initial preference data -> get $\pi_1$
2. **Round 2**: Use $\pi_1$ to generate new responses, rank them, create new preference data, train DPO -> get $\pi_2$
3. **Round 3**: Use $\pi_2$ to generate new responses, rank them, create new preference data, train DPO -> get $\pi_3$
4. ... repeat until diminishing returns

Each round uses **on-policy data** from the latest model, addressing the distribution mismatch.

### Why Iterate?

After one round of DPO, the model has improved. But the improvement reveals new failure modes
that were not in the original dataset. By generating new data from the improved model and
training again, you address these new failure modes.

In [ ]:
def iterative_dpo(rm: RewardModel, n_iterations: int = 5,
                  samples_per_iter: int = 300, dpo_epochs: int = 30,
                  input_dim: int = 16, output_dim: int = 32,
                  beta: float = 0.1) -> Dict:
    """Run iterative DPO: multiple rounds of generate -> rank -> train.
    
    Each iteration:
    1. Generate responses from current policy
    2. Score with reward model
    3. Create preference pairs
    4. Train DPO for several epochs
    5. Update reference policy to current policy
    """
    # Initialize policy
    policy = SimplePolicyNetwork(input_dim=input_dim, output_dim=output_dim).to(device)
    
    iteration_metrics = []
    all_losses = []
    all_accuracies = []
    
    for iteration in range(n_iterations):
        print(f"\n--- Iteration {iteration + 1}/{n_iterations} ---")
        
        # Set current policy as reference for this iteration
        ref_policy = copy.deepcopy(policy)
        ref_policy.eval()
        
        # Step 1: Generate on-policy data
        states = torch.randn(samples_per_iter, input_dim).to(device)
        
        with torch.no_grad():
            logits = policy(states)
            probs = F.softmax(logits, dim=-1)
            actions = torch.multinomial(probs, 4, replacement=True)  # 4 candidates
        
        # Step 2: Score with RM
        with torch.no_grad():
            scores = torch.zeros(samples_per_iter, 4).to(device)
            for j in range(4):
                scores[:, j] = rm(states, actions[:, j])
        
        # Step 3: Create preference pairs
        best_idx = scores.argmax(dim=1)
        worst_idx = scores.argmin(dim=1)
        chosen = actions.gather(1, best_idx.unsqueeze(1)).squeeze(1)
        rejected = actions.gather(1, worst_idx.unsqueeze(1)).squeeze(1)
        
        dataset = PreferenceDataset(states.cpu(), chosen.cpu(), rejected.cpu())
        
        # Step 4: Train DPO
        optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
        
        iter_losses = []
        iter_accs = []
        
        for epoch in range(dpo_epochs):
            epoch_loss = 0
            epoch_acc = 0
            n_batches = 0
            
            for s, c, r in dataloader:
                s, c, r = s.to(device), c.to(device), r.to(device)
                loss, acc = dpo_loss(policy, ref_policy, s, c, r, beta)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                epoch_acc += acc.item()
                n_batches += 1
            
            iter_losses.append(epoch_loss / n_batches)
            iter_accs.append(epoch_acc / n_batches)
        
        all_losses.extend(iter_losses)
        all_accuracies.extend(iter_accs)
        
        # Evaluate: average RM score of policy's top-1 action
        with torch.no_grad():
            eval_states = torch.randn(500, input_dim).to(device)
            eval_logits = policy(eval_states)
            eval_actions = eval_logits.argmax(dim=-1)
            eval_scores = rm(eval_states, eval_actions)
            avg_reward = eval_scores.mean().item()
        
        metrics = {
            "iteration": iteration + 1,
            "final_loss": iter_losses[-1],
            "final_accuracy": iter_accs[-1],
            "avg_reward": avg_reward,
        }
        iteration_metrics.append(metrics)
        print(f"  Loss: {metrics['final_loss']:.4f}, Acc: {metrics['final_accuracy']:.3f}, "
              f"Avg Reward: {metrics['avg_reward']:.3f}")
    
    return {
        "policy": policy,
        "iteration_metrics": iteration_metrics,
        "all_losses": all_losses,
        "all_accuracies": all_accuracies,
    }


# Run iterative DPO
print("ITERATIVE DPO: Multiple Rounds of Generate -> Rank -> Train")
print("=" * 60)

iter_result = iterative_dpo(
    rm, n_iterations=5, samples_per_iter=300,
    dpo_epochs=30, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM
)

In [ ]:
# Plot iterative DPO results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss over all epochs (with iteration boundaries)
axes[0].plot(iter_result['all_losses'], color='#3498db', linewidth=1.5)
epochs_per_iter = len(iter_result['all_losses']) // 5
for i in range(1, 5):
    axes[0].axvline(x=i * epochs_per_iter, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Total Epochs')
axes[0].set_ylabel('DPO Loss')
axes[0].set_title('DPO Loss Across Iterations')

# Accuracy per epoch
axes[1].plot(iter_result['all_accuracies'], color='#2ecc71', linewidth=1.5)
for i in range(1, 5):
    axes[1].axvline(x=i * epochs_per_iter, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Total Epochs')
axes[1].set_ylabel('Preference Accuracy')
axes[1].set_title('Accuracy Across Iterations')
axes[1].set_ylim(0.4, 1.0)

# Average reward per iteration
iterations = [m['iteration'] for m in iter_result['iteration_metrics']]
rewards = [m['avg_reward'] for m in iter_result['iteration_metrics']]
axes[2].bar(iterations, rewards, color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6'])
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Average Reward (RM Score)')
axes[2].set_title('Reward Improvement Per Iteration')

plt.tight_layout()
plt.savefig('iterative_dpo.png', dpi=100, bbox_inches='tight')
plt.show()

# Show improvement per iteration
print("\nIteration-by-Iteration Improvement:")
for m in iter_result['iteration_metrics']:
    print(f"  Iteration {m['iteration']}: Reward = {m['avg_reward']:.3f}, "
          f"Accuracy = {m['final_accuracy']:.3f}")

# Compute marginal improvement
print("\nMarginal improvement per iteration:")
for i in range(1, len(rewards)):
    delta = rewards[i] - rewards[i-1]
    print(f"  Iteration {i} -> {i+1}: {'+' if delta >= 0 else ''}{delta:.3f}")

print("\nNote: diminishing returns -- early iterations help most.")

## 6. Self-Play (SPIN)

**Paper**: Chen et al. 2024, ["Self-Play Fine-Tuning Converts Weak Language Models to Strong Language Models"](https://arxiv.org/abs/2401.01335)

### The Concept

SPIN (Self-Play fINe-tuning) frames alignment as a **two-player game**:

- **Main player**: The current model $\pi_{\theta}$ generates responses
- **Opponent**: The previous iteration's model $\pi_{\theta^{\text{old}}}$ also generates responses
- **Ground truth**: Human-written responses serve as the target distribution

The training objective:
- Prefer **human-written** responses over **model-generated** responses from the previous iteration
- This is DPO where chosen = human, rejected = model (from previous iteration)

### Why Self-Play?

1. **No reward model needed** -- just human reference data and model generations
2. **Natural convergence** -- when the model's distribution matches the human distribution, there is nothing left to learn (chosen = rejected in distribution)
3. **Progressive difficulty** -- as the model improves, the opponent (previous model) improves too, so the task gets harder

### SPIN Algorithm

```
For iteration t = 1, 2, 3, ...:
    1. Generate responses using pi_{t-1} (previous iteration model)
    2. Pair each (prompt, model_response) with (prompt, human_response)
    3. Train DPO: chosen = human, rejected = model_response
    4. pi_t = updated model
```

In [ ]:
class SPINTrainer:
    """Self-Play Fine-Tuning (SPIN).
    
    The model competes against its previous version. Human-written
    responses serve as the ground truth "expert" that the model
    tries to match.
    
    DPO pairs: chosen = human response, rejected = model response (from prev iteration)
    """
    
    def __init__(self, input_dim: int = 16, output_dim: int = 32,
                 hidden_dim: int = 64, beta: float = 0.1, lr: float = 1e-3):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.beta = beta
        self.lr = lr
        
        # Initialize policy
        self.policy = SimplePolicyNetwork(input_dim, hidden_dim, output_dim).to(device)
        # The "human" distribution is a fixed target policy (simulated)
        self.human_policy = SimplePolicyNetwork(input_dim, hidden_dim, output_dim).to(device)
        # Train human policy to be "good" (represents the target distribution)
        self._train_human_policy()
        self.human_policy.eval()
        
        self.iteration_metrics = []
    
    def _train_human_policy(self):
        """Create a 'human' policy that represents the target distribution.
        We train it on clean preference data to simulate human-quality responses."""
        clean_data, _ = generate_preference_data(
            n_samples=1000, noise_level=0.0, label_flip_prob=0.0,
            input_dim=self.input_dim, output_dim=self.output_dim
        )
        ref = copy.deepcopy(self.human_policy)
        ref.eval()
        optimizer = torch.optim.Adam(self.human_policy.parameters(), lr=1e-3)
        dataloader = DataLoader(clean_data, batch_size=32, shuffle=True)
        
        for epoch in range(50):
            for states, chosen, rejected in dataloader:
                states = states.to(device)
                chosen = chosen.to(device)
                rejected = rejected.to(device)
                loss, _ = dpo_loss(self.human_policy, ref, states, chosen, rejected, self.beta)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
    def sample_human_action(self, states: torch.Tensor) -> torch.Tensor:
        """Sample from the 'human' policy."""
        with torch.no_grad():
            logits = self.human_policy(states)
            probs = F.softmax(logits, dim=-1)
            return torch.multinomial(probs, 1).squeeze(1)
    
    def sample_model_action(self, policy: SimplePolicyNetwork, 
                           states: torch.Tensor) -> torch.Tensor:
        """Sample from a given policy."""
        with torch.no_grad():
            logits = policy(states)
            probs = F.softmax(logits, dim=-1)
            return torch.multinomial(probs, 1).squeeze(1)
    
    def spin_iteration(self, n_samples: int = 300, n_epochs: int = 30) -> Dict:
        """Run one SPIN iteration.
        
        1. Save current policy as 'previous'
        2. Generate responses from previous policy
        3. Get human responses for the same prompts
        4. DPO: chosen=human, rejected=previous_model
        """
        # Step 1: Save previous policy
        prev_policy = copy.deepcopy(self.policy)
        prev_policy.eval()
        
        # Reference policy for DPO (frozen copy of current)
        ref_policy = copy.deepcopy(self.policy)
        ref_policy.eval()
        
        # Step 2 & 3: Generate states, get human and model responses
        states = torch.randn(n_samples, self.input_dim).to(device)
        human_actions = self.sample_human_action(states)  # chosen
        model_actions = self.sample_model_action(prev_policy, states)  # rejected
        
        # Step 4: Train DPO
        dataset = PreferenceDataset(states.cpu(), human_actions.cpu(), model_actions.cpu())
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
        optimizer = torch.optim.Adam(self.policy.parameters(), lr=self.lr)
        
        losses = []
        accuracies = []
        
        for epoch in range(n_epochs):
            epoch_loss = 0
            epoch_acc = 0
            n_batches = 0
            
            for s, c, r in dataloader:
                s, c, r = s.to(device), c.to(device), r.to(device)
                loss, acc = dpo_loss(self.policy, ref_policy, s, c, r, self.beta)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                epoch_acc += acc.item()
                n_batches += 1
            
            losses.append(epoch_loss / n_batches)
            accuracies.append(epoch_acc / n_batches)
        
        # Evaluate: KL divergence between model and human
        with torch.no_grad():
            eval_states = torch.randn(500, self.input_dim).to(device)
            model_logits = self.policy(eval_states)
            human_logits = self.human_policy(eval_states)
            
            model_probs = F.softmax(model_logits, dim=-1)
            human_probs = F.softmax(human_logits, dim=-1)
            
            # KL(human || model)
            kl = F.kl_div(model_probs.log(), human_probs, reduction='batchmean').item()
            
            # Agreement: how often do model and human pick the same action?
            model_top = model_logits.argmax(dim=-1)
            human_top = human_logits.argmax(dim=-1)
            agreement = (model_top == human_top).float().mean().item()
        
        metrics = {
            "final_loss": losses[-1],
            "final_accuracy": accuracies[-1],
            "kl_divergence": kl,
            "agreement_with_human": agreement,
            "losses": losses,
            "accuracies": accuracies,
        }
        self.iteration_metrics.append(metrics)
        return metrics
    
    def run(self, n_iterations: int = 5, n_samples: int = 300, 
            n_epochs: int = 30) -> List[Dict]:
        """Run SPIN for multiple iterations."""
        for i in range(n_iterations):
            print(f"\n--- SPIN Iteration {i+1}/{n_iterations} ---")
            metrics = self.spin_iteration(n_samples, n_epochs)
            print(f"  Loss: {metrics['final_loss']:.4f}, "
                  f"KL(human||model): {metrics['kl_divergence']:.4f}, "
                  f"Agreement: {metrics['agreement_with_human']:.3f}")
        
        return self.iteration_metrics


# Run SPIN
print("SPIN: Self-Play Fine-Tuning")
print("=" * 50)
print("Model learns by competing against its previous version.")
print("Chosen = human response, Rejected = previous model response.")
print("Converges when model distribution matches human distribution.\n")

spin_trainer = SPINTrainer(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM)
spin_metrics = spin_trainer.run(n_iterations=5, n_samples=300, n_epochs=30)

In [ ]:
# Plot SPIN results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

iterations = list(range(1, len(spin_metrics) + 1))

# KL divergence over iterations (should decrease)
kls = [m['kl_divergence'] for m in spin_metrics]
axes[0].plot(iterations, kls, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[0].set_xlabel('SPIN Iteration')
axes[0].set_ylabel('KL(human || model)')
axes[0].set_title('KL Divergence from Human Distribution')
axes[0].set_xticks(iterations)

# Agreement with human policy (should increase)
agreements = [m['agreement_with_human'] for m in spin_metrics]
axes[1].plot(iterations, agreements, 'o-', color='#2ecc71', linewidth=2, markersize=8)
axes[1].set_xlabel('SPIN Iteration')
axes[1].set_ylabel('Agreement with Human')
axes[1].set_title('Model-Human Agreement Rate')
axes[1].set_xticks(iterations)
axes[1].set_ylim(0, 1)

# DPO accuracy within each iteration (should start high and stay high)
final_accs = [m['final_accuracy'] for m in spin_metrics]
axes[2].bar(iterations, final_accs, 
            color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#9b59b6'])
axes[2].set_xlabel('SPIN Iteration')
axes[2].set_ylabel('DPO Preference Accuracy')
axes[2].set_title('Within-Iteration DPO Accuracy')
axes[2].set_xticks(iterations)
axes[2].set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig('spin_results.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key observations:")
print("  1. KL divergence should decrease as model approaches human distribution")
print("  2. Agreement with human should increase")
print("  3. DPO accuracy may decrease in later iterations (because the model's")
print("     responses become more similar to human -- harder to distinguish)")
print("\nThis is the SPIN convergence criterion: when the model IS the human,")
print("there's nothing left to learn.")

## 7. Reward Model Overoptimization

**Paper**: Gao et al. 2023, ["Scaling Laws for Reward Model Overoptimization"](https://arxiv.org/abs/2210.10760)

### The Problem

When you optimize a policy against a reward model, two things happen:

1. **Proxy reward** (the RM's score) goes up continuously
2. **True reward** (actual human preference) goes up initially, then **goes back down**

This is **Goodhart's Law** applied to alignment: "When a measure becomes a target, it ceases to be a good measure."

### The Scaling Law

Gao et al. define the distance $d = \sqrt{\text{KL}(\pi \| \pi_{\text{init}})}$ (note: $d$ is the distance variable, **not** a coefficient) and fit:
- Best-of-$n$: $R_{\text{bon}}(d) = d(\alpha_{\text{bon}} - \beta_{\text{bon}} d)$
- RL: $R_{\text{RL}}(d) = d(\alpha_{\text{RL}} - \beta_{\text{RL}} \log d)$
- The $\alpha$ term captures genuine improvement; the $\beta$ term captures overoptimization
- The coefficients vary smoothly (approximately log-linearly) with reward-model size
- At small $d$ (low KL budgets), genuine improvement dominates; at large $d$, overoptimization dominates

### The Practical Implication

There is an **optimal KL budget** beyond which more optimization hurts. This means:

1. You need to **stop training** before the proxy reward plateaus
2. **KL penalty** in PPO serves as regularization against overoptimization
3. **DPO's beta** parameter implicitly controls the KL budget
4. **RM quality** determines how far you can optimize before overoptimizing

In [ ]:
# Demonstrate reward model overoptimization

def simulate_overoptimization(alpha: float = 3.0, beta_coef: float = 0.5,
                              max_kl: float = 30.0,
                              n_points: int = 200) -> Dict:
    """Simulate the overoptimization scaling law from Gao et al. 2023.
    
    Following the paper, define the DISTANCE d = sqrt(KL(pi || pi_init)).
    The fitted forms are:
        Best-of-n: R_bon(d) = d * (alpha_bon - beta_bon * d)
        RL:        R_RL(d)  = d * (alpha_RL  - beta_RL * log d)
    where the alpha/beta coefficients vary smoothly (roughly log-linearly)
    with reward-model size. (d is the distance variable, NOT a coefficient.)
    We simulate the best-of-n form: gold(d) = alpha*d - beta_coef*d^2.
    
    alpha controls the rate of genuine improvement
    beta_coef controls the rate of overoptimization
    """
    kl_values = np.linspace(0.01, max_kl, n_points)
    d_values = np.sqrt(kl_values)  # d = sqrt(KL), per the paper
    
    gold_reward = d_values * (alpha - beta_coef * d_values)
    # NOTE: the proxy curve is an ILLUSTRATIVE monotone stand-in -- Gao et al. do NOT
    # fit a closed form for the proxy. Only the GOLD form d*(alpha - beta*d) is from
    # the paper. We just need a curve that keeps rising, which is how an over-optimized
    # RM score behaves in practice.
    proxy_reward = d_values * (alpha + 0.3 * d_values)  # RM score: only ever rises
    
    # Optimal d: d/dd [alpha*d - beta_coef*d^2] = 0  =>  d* = alpha / (2*beta_coef)
    optimal_d = alpha / (2 * beta_coef)
    optimal_kl = optimal_d ** 2
    optimal_gold = optimal_d * (alpha - beta_coef * optimal_d)
    
    return {
        "kl_values": kl_values,
        "gold_reward": gold_reward,
        "proxy_reward": proxy_reward,
        "optimal_kl": optimal_kl,
        "optimal_gold": optimal_gold,
    }


# Plot overoptimization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Gold vs Proxy reward
result = simulate_overoptimization(alpha=3.0, beta_coef=0.5)

axes[0].plot(result['kl_values'], result['proxy_reward'], 
             label='Proxy Reward (RM score)', color='#e74c3c', linewidth=2, linestyle='--')
axes[0].plot(result['kl_values'], result['gold_reward'],
             label='Gold Reward (true quality)', color='#2ecc71', linewidth=2)
axes[0].axvline(x=result['optimal_kl'], color='gray', linestyle=':', alpha=0.7)
axes[0].annotate(f"Optimal KL = {result['optimal_kl']:.1f}",
                xy=(result['optimal_kl'], result['optimal_gold']),
                xytext=(result['optimal_kl'] + 2, result['optimal_gold'] + 1),
                arrowprops=dict(arrowstyle='->', color='gray'),
                fontsize=10)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].set_xlabel('KL Divergence from Reference Policy')
axes[0].set_ylabel('Reward')
axes[0].set_title('Reward Model Overoptimization')
axes[0].legend()
axes[0].fill_between(result['kl_values'], 0, result['gold_reward'],
                     where=result['gold_reward'] > 0, alpha=0.1, color='green')
axes[0].fill_between(result['kl_values'], 0, result['gold_reward'],
                     where=result['gold_reward'] < 0, alpha=0.1, color='red')

# Panel 2: Effect of RM quality (different d/c ratios)
for a, b, label in [(3.0, 0.3, 'Strong RM (alpha/beta=10)'), 
                     (3.0, 0.5, 'Medium RM (alpha/beta=6)'),
                     (3.0, 1.0, 'Weak RM (alpha/beta=3)')]:
    res = simulate_overoptimization(alpha=a, beta_coef=b)
    axes[1].plot(res['kl_values'], res['gold_reward'], label=label, linewidth=2)
    axes[1].axvline(x=res['optimal_kl'], color='gray', linestyle=':', alpha=0.3)

axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_xlabel('KL Divergence from Reference Policy')
axes[1].set_ylabel('Gold Reward')
axes[1].set_title('RM Quality Determines Optimization Budget')
axes[1].legend()

plt.tight_layout()
plt.savefig('rm_overoptimization.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key takeaways:")
print("  1. Proxy reward (RM score) always increases -- it LIES to you")
print("  2. Gold reward (true quality) peaks then declines")
print("  3. Better RMs allow more optimization before overoptimizing")
print("  4. The KL penalty (beta in DPO, KL coefficient in PPO) is your protection")
print(f"\n  Optimal KL for medium RM: {result['optimal_kl']:.1f}")
print(f"  Maximum achievable gold reward: {result['optimal_gold']:.2f}")

### Practical Mitigations for Overoptimization

1. **KL budget / early stopping**: Monitor KL divergence from reference, stop when budget exhausted
2. **RM ensembles**: Use multiple RMs, take the minimum or average score. Harder to hack an ensemble.
3. **Iterative re-labeling**: Periodically re-evaluate with humans (or a different judge) to catch drift
4. **Conservative optimization**: Use larger beta in DPO, larger KL coefficient in PPO
5. **Best-of-N instead of RL**: Rejection sampling (generate N, keep best) is less prone to overoptimization than gradient-based RL because it does not update the model's parameters

## 8. The Llama 3 Post-Training Recipe

Meta's Llama 3 paper (2024) provides one of the most detailed public descriptions of an
industrial-scale post-training pipeline. Here is the recipe:

### Stage 1: Supervised Fine-Tuning (SFT)
- Curated, high-quality instruction-following data
- Focus on data quality over quantity
- Include multi-turn conversations, code, math, reasoning

### Stage 2: Rejection Sampling
- For each prompt, generate N responses (e.g., N=10-30)
- Score each response with a reward model
- Keep only the top-K (e.g., K=1) responses
- The selected responses become **SFT data** for the next round (alongside curated and synthetic data) -- not DPO pairs
- This is **simpler than PPO** and surprisingly competitive

### Stage 3: DPO on Human Preference Pairs
- DPO trains on **human-annotated preference pairs** collected on responses from recent model versions (not on rejection-sampled best-vs-worst pairs)
- This gives further improvement beyond SFT alone

### Stage 4: Iterate
- Repeat the round of (rejection sampling -> SFT -> DPO) multiple times -- the paper describes roughly six rounds
- Each iteration uses the latest model(s) for generation
- Safety SFT and preference data are integrated across the rounds, not saved for a separate final pass

### Key Lessons from Llama 3

1. **Simple methods at scale > complex methods at small scale**: Rejection sampling + DPO > PPO for them
2. **Data quality is king**: They invested heavily in curation, not just collection
3. **Iterative training is essential**: Single-round post-training is suboptimal
4. **The base model determines the ceiling**: Post-training cannot add capabilities the base model lacks

### The Decision Framework

```
IF you have a strong base model AND high-quality preference data:
    -> Start with DPO (simplest, most stable)
    -> Try iterative DPO for additional gains
    -> Consider online DPO if you have a good reward model

IF you need maximum performance AND have engineering resources:
    -> PPO/GRPO with a well-trained reward model
    -> KL-constrained, with careful hyperparameter tuning
    -> Monitor for overoptimization

IF you have limited data:
    -> Generate synthetic preferences (Constitutional AI, Best-of-N)
    -> Prioritize quality over quantity
    -> Use SPIN if you have gold SFT data but no preference data

IF you are at Meta/OpenAI/Anthropic/DeepMind scale:
    -> Iterative rounds of SFT (on curated + rejection-sampled data) then DPO (on human preference pairs), ~6 rounds (Llama 3 recipe)
    -> Or: PPO/GRPO with massive RL compute (DeepSeek-R1 approach)
    -> Both work; the choice depends on engineering team expertise
```

**Insider Tip:** The Llama 3 recipe (SFT -> rejection sampling -> DPO -> iterate) is the current industry standard. Meta published this openly. In an interview, being able to describe this recipe end-to-end shows you understand production post-training. The key insight is that rejection sampling is simpler than PPO but achieves similar results when you have a good reward model. The iteration is what makes it work -- each round uses the improved model to generate harder training data. If asked "how would you post-train a model?", this recipe is the answer most frontier labs want to hear.

In [ ]:
# Simulate the Llama 3 recipe: Rejection Sampling + DPO, iterated

def rejection_sampling_dpo(rm: RewardModel, n_iterations: int = 4,
                           n_prompts: int = 200, n_candidates: int = 8,
                           dpo_epochs: int = 30,
                           input_dim: int = 16, output_dim: int = 32,
                           beta: float = 0.1) -> Dict:
    """Toy "rejection sampling + DPO" loop (loosely Llama 3-inspired).
    
    For each prompt:
    1. Generate N candidates from current policy
    2. Score with RM
    3. Best = chosen, Worst = rejected
    4. Train DPO on these pairs
    5. Repeat
    
    NOTE: the actual Llama 3 recipe differs -- rejection sampling builds SFT
    data (the best responses are folded into the SFT mix), while DPO trains
    on HUMAN-annotated preference pairs, iterated over ~6 rounds. The toy
    best-vs-worst DPO pairs here are a simplification.
    """
    policy = SimplePolicyNetwork(input_dim=input_dim, output_dim=output_dim).to(device)
    
    all_metrics = []
    all_rewards = []
    
    for iteration in range(n_iterations):
        print(f"\n--- Llama 3 Recipe: Iteration {iteration+1}/{n_iterations} ---")
        
        ref_policy = copy.deepcopy(policy)
        ref_policy.eval()
        
        # Step 1: Generate candidates
        states = torch.randn(n_prompts, input_dim).to(device)
        with torch.no_grad():
            logits = policy(states)
            probs = F.softmax(logits, dim=-1)
            candidates = torch.multinomial(probs, n_candidates, replacement=True)
        
        # Step 2: Score with RM
        with torch.no_grad():
            scores = torch.zeros(n_prompts, n_candidates).to(device)
            for j in range(n_candidates):
                scores[:, j] = rm(states, candidates[:, j])
        
        # Step 3: Create preference pairs
        best_idx = scores.argmax(dim=1)
        worst_idx = scores.argmin(dim=1)
        chosen = candidates.gather(1, best_idx.unsqueeze(1)).squeeze(1)
        rejected = candidates.gather(1, worst_idx.unsqueeze(1)).squeeze(1)
        
        # Step 4: DPO training
        dataset = PreferenceDataset(states.cpu(), chosen.cpu(), rejected.cpu())
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
        optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
        
        for epoch in range(dpo_epochs):
            for s, c, r in dataloader:
                s, c, r = s.to(device), c.to(device), r.to(device)
                loss, acc = dpo_loss(policy, ref_policy, s, c, r, beta)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
        # Evaluate
        with torch.no_grad():
            eval_states = torch.randn(500, input_dim).to(device)
            eval_logits = policy(eval_states)
            eval_actions = eval_logits.argmax(dim=-1)
            eval_scores = rm(eval_states, eval_actions)
            avg_reward = eval_scores.mean().item()
        
        all_rewards.append(avg_reward)
        print(f"  Avg Reward: {avg_reward:.3f}")
    
    return {"rewards": all_rewards, "policy": policy}


print("LLAMA 3 RECIPE: Rejection Sampling + DPO (Iterated)")
print("=" * 60)

llama3_result = rejection_sampling_dpo(
    rm, n_iterations=4, n_prompts=200, n_candidates=8,
    dpo_epochs=30, input_dim=INPUT_DIM, output_dim=OUTPUT_DIM
)

In [ ]:
# Compare all methods
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Iterative DPO rewards
iter_rewards = [m['avg_reward'] for m in iter_result['iteration_metrics']]

# SPIN agreement (as proxy for quality)
spin_agreements = [m['agreement_with_human'] for m in spin_metrics]

# Llama 3 rewards
llama3_rewards = llama3_result['rewards']

# Normalize all to [0, 1] for fair comparison
def normalize(arr):
    arr = np.array(arr)
    if arr.max() == arr.min():
        return np.ones_like(arr) * 0.5
    return (arr - arr.min()) / (arr.max() - arr.min())

x_iter = range(1, len(iter_rewards) + 1)
x_llama = range(1, len(llama3_rewards) + 1)
x_spin = range(1, len(spin_agreements) + 1)

ax.plot(x_iter, normalize(iter_rewards), 'o-', label='Iterative DPO', 
        color='#3498db', linewidth=2, markersize=8)
ax.plot(x_llama, normalize(llama3_rewards), 's-', label='Llama 3 Recipe (RS + DPO)',
        color='#e67e22', linewidth=2, markersize=8)
ax.plot(x_spin, normalize(spin_agreements), '^-', label='SPIN (agreement)',
        color='#2ecc71', linewidth=2, markersize=8)

ax.set_xlabel('Iteration')
ax.set_ylabel('Normalized Quality (0 = worst, 1 = best)')
ax.set_title('Comparison: Iterative DPO vs Llama 3 Recipe vs SPIN')
ax.legend()
ax.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.savefig('method_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("All three iterative methods show improvement over iterations,")
print("with diminishing returns. The choice between them depends on:")
print("  - Available data (SPIN needs gold SFT data, others need RM)")
print("  - Engineering complexity (rejection sampling is simplest)")
print("  - Scale (Llama 3 recipe works best at large scale)")

## 9. "Why Does This Work?" -- Critical Analysis

### Why Does Online Beat Offline?

The mathematical answer: DPO's loss function assumes the preference data comes from the
same distribution as the policy. In offline DPO, this assumption is violated because the
data was generated by the *initial* policy, but we train a *different* policy. The gradient
direction becomes less accurate as the policy diverges from the data-generating distribution.

The intuitive answer: imagine learning to play chess from a fixed set of games by a beginner.
After you improve, those beginner-level games no longer teach you anything useful. You need
games at your current level (online) to keep improving.

### When Should You Use RLHF vs DPO vs Rejection Sampling?

| Method | Strengths | Weaknesses | Best For |
|---|---|---|---|
| **RLHF (PPO)** | Can scale RL compute, supports exploration | Complex, unstable, expensive | Maximum performance, large teams |
| **DPO** | Simple, stable, one-stage | Offline only (unless iterated), no exploration | Most scenarios, good default choice |
| **Rejection Sampling + DPO** | Very simple, robust, parallelizable | Requires strong RM, wasteful (generates N, keeps 1) | Industrial scale (Llama 3) |
| **SPIN** | No RM needed, natural convergence | Requires gold SFT data, limited by SFT data quality | When you have gold data but no preferences |
| **Online DPO** | Best of both: simple + on-policy | Requires RM for ranking | When you have a good RM |

### Decision Framework

1. **What data do you have?**
   - Human preference pairs? -> DPO
   - Gold SFT data only? -> SPIN
   - A trained reward model? -> Rejection sampling or online DPO
   - Nothing? -> Generate synthetic data first (Notebook 11)

2. **What is your compute budget?**
   - Limited? -> Offline DPO (one pass over data)
   - Moderate? -> Iterative DPO (3-5 rounds)
   - Large? -> Online DPO or PPO/GRPO

3. **What is your engineering bandwidth?**
   - Small team? -> DPO or rejection sampling
   - Large team? -> PPO with the full infrastructure

---
## Interview Question Bank: Scaling Post-Training

*These questions are for Principal-level candidates and team leads. They test whether you can own an entire post-training pipeline -- not just implement one component, but design, execute, evaluate, and iterate the full system. If you are targeting Senior, prepare Q2. If you are targeting Principal, prepare all three.*

---

**Q1: "You are leading the post-training team for a new frontier model. Describe your approach from SFT to final model."**

**What we're testing:** Leadership-level system thinking, ability to own a multi-month program, risk management, decision-making under uncertainty. This is a PRINCIPAL-LEVEL question.

**Good answer:**
- Describes the standard pipeline: SFT -> reward model training -> DPO/RLHF -> evaluation -> iteration
- Mentions key decisions at each stage: data selection, hyperparameters, evaluation criteria
- Has a timeline: "This takes 2-4 months for a frontier model"

**Great answer (Principal level):**

"I would follow the Llama 3 recipe as a foundation, then customize based on our model's characteristics and target use case. Here is my plan:

**Phase 1: SFT Foundation (Weeks 1-3)**
- Data: curate 100K-500K high-quality instruction-response pairs. Mix of human-written (30%) and synthetic (70%). Diverse across tasks, difficulty levels, and languages.
- Training: 2-3 epochs, learning rate warmup then cosine decay. Validate on a held-out set every 1K steps.
- Evaluation gate: model must pass instruction-following benchmarks (IFEval) and not regress on core capabilities (MMLU, HumanEval). If it fails, debug data quality before proceeding.

**Phase 2: Reward Model Training (Weeks 2-4, parallel with Phase 1)**
- Train a reward model on 100K-500K human preference pairs
- Validate RM accuracy on a held-out preference set (target: >70% agreement with humans)
- Train a separate safety RM for safety-specific evaluation
- Also train a process reward model (PRM) if the model will be used for reasoning tasks

**Phase 3: Alignment Iteration (Weeks 4-8)**
- Round 1: DPO on initial preference data (offline). Quick, stable, establishes alignment baseline.
- Round 2: Rejection sampling with RM + DPO on re-ranked data (semi-online). Improves over Round 1.
- Round 3-4: On-policy DPO or GRPO for specific capabilities (reasoning, coding). This is where the major quality gains happen.
- Each round: evaluate against the previous round on ALL metrics (helpfulness, safety, capabilities, diversity). Only proceed if the new model is strictly better or better on target metrics without regression on others.

**Phase 4: Safety & Red-Teaming (Weeks 6-10, parallel with Phase 3)**
- Automated safety testing: harmbench, red-team prompt suites
- Human red-teaming: dedicated team trying to break the model
- Constitutional AI pass: safety-specific alignment using constitutional principles
- Evaluation gate: model must pass safety thresholds BEFORE shipping. This is non-negotiable.

**Phase 5: Final Evaluation & Launch (Weeks 9-12)**
- Comprehensive human evaluation: 1000+ A/B comparisons against the previous model version
- Capability regression testing: ensure no capabilities were lost during alignment
- Deployment planning: inference optimization, monitoring setup, rollback plan

**Team structure:**
- 2-3 researchers on data curation and synthetic data
- 2-3 researchers on training (SFT, DPO, GRPO)
- 1-2 researchers on reward modeling and evaluation
- 1 researcher on safety and red-teaming
- Total: 6-9 researchers for 3 months. This is a $2-5M program (compute + people)."

**Red flag:** "I would just do SFT and DPO." No iteration, no evaluation gates, no safety considerations, no team planning. Shows no awareness of the complexity of a real post-training program.

**Follow-up:** "Your RM score is improving but human eval is flat. What do you do?"
- Expected: This is reward model overoptimization (Goodhart's Law). The model is learning to game the RM without actually improving quality. Solutions: (1) retrain the RM on more diverse data, (2) add KL penalty to prevent over-optimization, (3) switch to a different evaluation signal (human eval directly), (4) use an ensemble of RMs instead of a single RM, (5) inspect the model's outputs qualitatively -- what is it doing differently that pleases the RM but not humans? Common cause: RM rewards length, hedging, or sycophancy, which do not improve actual quality.

---

**Q2: "Online vs offline DPO -- explain the trade-off quantitatively."**

**What we're testing:** Understanding of the distribution mismatch problem, ability to reason quantitatively about ML trade-offs, awareness of when approximations are sufficient.

**Good answer:**
- Online DPO generates new data from the current policy, so preferences are always on-distribution
- Offline DPO uses a fixed preference dataset, which becomes off-distribution as the policy changes during training
- Online is more expensive (generation cost) but produces better alignment
- Offline is cheaper and faster but bounded by the quality of the initial dataset

**Great answer (Senior -> Principal level):**
- Explains the distribution mismatch: "After k steps of offline DPO, the policy pi_k has diverged from the policy pi_0 that generated the preference data. The effective 'staleness' of the data grows with KL(pi_k || pi_0); as the divergence grows, the offline data becomes increasingly misleading -- the model is learning preferences about outputs it would no longer generate. There is no universal KL threshold for when this bites; it is setup-dependent."
- Discusses the compute overhead: "Online DPO requires generating N completions per prompt, scoring them, and constructing new preference pairs. This adds substantial wall-clock time per iteration -- how much depends on generation lengths, batch sizes, and serving infrastructure. Published comparisons generally show meaningful win-rate gains from on-policy data, but the exact gain is setup-dependent; for frontier models the cost is usually judged worth it."
- Discusses the iterative compromise: "The practical middle ground is iterative DPO: run a few rounds of offline DPO, regenerating preference data between rounds. Each round restarts from fresh on-policy data, so the staleness resets. This captures much of the benefit of fully online DPO at substantially lower cost, though the exact tradeoff is setup-dependent."
- Knows when offline is good enough: "If the base model is already close to the target distribution (e.g., a strong SFT model being fine-tuned for style), offline DPO is sufficient. The distribution mismatch is small because the policy does not need to move far. For capability elicitation (reasoning, complex instruction following), online methods are essential."
- References the Llama 3 approach: "Meta uses iterative rejection sampling + DPO, which is their version of the online-offline compromise. They report 6 rounds of iterative DPO with on-policy data refresh, and each round contributes meaningful improvement."

**Red flag:** "Online is always better." Cannot quantify the trade-off. Does not know what distribution mismatch means in this context.

**Follow-up:** "At what point does the online overhead stop being worth it? How would you decide?"
- Expected: Track the marginal improvement per iteration. Plot win rate vs iteration number. When the marginal gain drops below a threshold (e.g., <1% win rate improvement per round), stop. Also: compare the cost of one more DPO iteration vs spending that compute on other improvements (better data, larger model, more safety testing).

---

**Q3: "Your model has become sycophantic -- it agrees with everything the user says, even when the user is wrong. How do you fix this?"**

**What we're testing:** Diagnosis of a real production failure mode, knowledge of the sycophancy problem, multi-faceted solution thinking.

**Good answer:**
- Sycophancy is a known failure mode of RLHF/DPO -- the model learns that agreeing with the user gets higher preference ratings
- The root cause is in the preference data: annotators tend to prefer agreeable responses
- Fix: add anti-sycophancy examples to the preference data (responses that politely disagree with incorrect user claims should be preferred)

**Great answer (Senior -> Principal level):**
- Diagnoses the multi-level problem:
  - **Data level**: Preference annotators have a bias toward agreeable responses. Need to explicitly instruct annotators to prefer accuracy over agreeableness, or use expert annotators for factual domains.
  - **Algorithm level**: DPO/RLHF optimizes for human preferences, and humans have a systematic preference for validation. Need to add a "factual accuracy" signal that is independent of preference ratings.
  - **Evaluation level**: Standard eval may not catch sycophancy because the evaluator (human or AI) may also prefer agreeable responses. Need specific sycophancy benchmarks (e.g., present the model with a false claim and measure whether it pushes back).
- Concrete fixes:
  1. Create a sycophancy evaluation set: 500+ prompts where the user states something factually wrong. Score the model on whether it corrects the user.
  2. Add constitutional AI principle: "When the user states something factually incorrect, politely correct them rather than agreeing."
  3. Add anti-sycophancy preference pairs to the training data: (user says wrong thing, model corrects = chosen) vs (user says wrong thing, model agrees = rejected)
  4. Train a "sycophancy detector" classifier and use it as a negative reward signal during RLHF
- Notes the tension: "There is a real trade-off between helpfulness and truthfulness. A model that always disagrees is unhelpful. A model that always agrees is untruthful. The goal is calibrated disagreement -- disagree on facts, accommodate on preferences."

**Red flag:** "Just tell the model to be honest in the system prompt." Does not understand that sycophancy is a training artifact, not an inference problem.

---
## Production Implementation Notes: Scaling Post-Training

### The Llama 3 Recipe in Detail (The Industry Standard)

This is the recipe every frontier lab uses as a starting point (with their own modifications). Know it cold.

```
Llama 3 Post-Training Pipeline (one round; the paper iterates this ~6 times):
1. Reward Model training on human-annotated preference pairs
2. Rejection Sampling: generate 10-30 responses per prompt, select top-k by RM score
3. SFT on curated + synthetic + rejection-sampled data
4. DPO on human-annotated preference pairs (collected on recent model responses)
5. Repeat steps 1-4 for ~6 rounds, refreshing data with the latest models
6. Safety: safety SFT and preference data are integrated across the rounds (no separate final pass)
7. Final evaluation: human eval + automated benchmarks + safety testing
```

Key details that matter in practice:
- **Step 2 (rejection sampling)** is the most compute-intensive step. Generating 30 completions per prompt for 100K prompts at 70B scale requires thousands of GPU-hours.
- **Step 5 (iteration)** has diminishing returns. Rounds 1-3 provide the largest gains. Rounds 4-6 provide smaller but still meaningful improvements. After 6 rounds, the gains are typically negligible.
- **Safety (step 6)**: in the Llama 3 paper, safety SFT and safety preference data are integrated across the post-training rounds (rather than applied as one final pass), with safety evaluations run each round to catch regressions early.

### Reward Model Overoptimization: The Quantitative Story

From Gao et al. (2023), "Scaling Laws for Reward Model Overoptimization":

- The reward model score (RM proxy) increases monotonically as you optimize against it
- The true reward (human judgment) increases initially, peaks, then DECREASES as optimization continues
- In the paper's fitted forms, define d = sqrt(KL(pi || pi_init)) -- d is the distance variable, not a coefficient. Gold reward follows R(d) = d(alpha_bon - beta_bon*d) for best-of-n and R(d) = d(alpha_RL - beta_RL*log d) for RL, with the coefficients varying smoothly (roughly log-linearly) with RM size
- **Practical implication**: set a KL budget BEFORE training and stop when KL(pi || pi_ref) exceeds it. The right budget depends on RM quality and setup -- calibrate it empirically (e.g., where held-out human eval peaks) rather than relying on a universal number.

### Compute Budget Allocation: Where to Spend Your FLOPs

For a fixed post-training compute budget, here is the rough allocation that works best:

| Component | % of Budget | Why |
|-----------|-------------|-----|
| Data curation + synthetic generation | 10-15% | Higher quality data multiplies the value of all other compute |
| SFT | 5-10% | Quick, usually 2-3 epochs |
| Reward model training | 5-10% | Important but not compute-heavy relative to policy training |
| Alignment training (DPO/GRPO) | 40-50% | The core of post-training. Multiple iterations needed. |
| Evaluation + safety testing | 15-20% | Under-invested by most teams. Budget for human eval. |
| Experiments / exploration | 10-15% | Try new things. This is where breakthroughs come from. |

### Monitoring and Observability

What to track during post-training (in order of importance):

1. **Loss curves** (obvious, but many teams do not check for divergence early enough)
2. **KL divergence from reference** (the most important single metric -- if KL explodes, the model is going off-distribution)
3. **Response length distribution** (length creep = length exploitation)
4. **RM score on held-out set** (not the training RM -- a separate evaluation RM)
5. **Safety scores on fixed benchmark** (track regression)
6. **Generation diversity** (distinct-n, self-BLEU -- track mode collapse)
7. **Qualitative spot checks** (read 10-20 random outputs every day. There is no substitute for human eyes.)

---
## How Scaling Post-Training Gets Tested in Interviews

### This Is the "Director-Level" Topic

Scaling post-training questions are primarily asked at the Principal level and above. They test whether you can own a program, not just execute a task. The key distinction:

| Level | What They Test | Example Question |
|-------|---------------|------------------|
| **Senior** | "Can you implement DPO?" | "Implement the DPO loss and train on this dataset." |
| **Principal** | "Can you design a post-training pipeline?" | "You are leading post-training for a new 70B model. Walk me through your plan." |
| **Director** | "Can you make the hard trade-offs?" | "We have 3 months and $5M. Where do you invest and what do you cut?" |

### The "Pipeline Design" Interview

This is a 45-60 minute whiteboard session where you design a full post-training pipeline. The interviewer gives you constraints and then probes your decisions:

**Round 1 (15 min):** "Describe your approach." -- High-level architecture.
**Round 2 (15 min):** "What if X goes wrong?" -- The interviewer introduces failure scenarios (RM overoptimization, safety regression, compute overrun). Can you adapt?
**Round 3 (15 min):** "How do you evaluate success?" -- Evaluation methodology, go/no-go criteria, risk assessment.

### Preparation Strategy for Principal-Level Candidates

1. **Know the Llama 3 recipe cold.** Section 4 of the Llama 3 paper is your bible. Be able to recite the pipeline and explain every decision.
2. **Know the numbers.** How much does post-training cost? How many GPU-hours? How many preference pairs? How many rounds? Approximate numbers are fine -- the point is that you have calibrated intuitions.
3. **Know the failure modes.** RM overoptimization, sycophancy, length exploitation, safety regression, capability loss. For each, know the diagnosis AND the fix.
4. **Have opinions.** "The Llama 3 recipe is good but I would change X because Y." Having a well-reasoned opinion (even if the interviewer disagrees) is better than reciting the standard approach. This is the Principal-level signal.
5. **Practice the timeline question.** "How long does this take? What is the critical path? Where can you parallelize?" If you cannot estimate a timeline, you cannot lead a program.

## 10. Flashcard Summary

Print these, review them daily.

---

| # | Question | Answer |
|---|---|---|
| 1 | Do scaling laws apply to post-training? | Yes, but differently. More RL compute can help (DeepSeek-R1), but data quality matters more than quantity, and there is an overoptimization ceiling determined by RM quality. |
| 2 | What is the Phi-1 lesson for post-training? | Per-example data quality dominates: at a FIXED budget, clean preference pairs beat noisy ones (in our demo ~0.91 vs ~0.72 gold). Caveat: quality is not a substitute for quantity -- a tiny clean set still loses to a large one. Invest in curation AND enough of it. |
| 3 | Online vs offline DPO: which wins and why? | Online DPO wins because it avoids distribution mismatch. Offline trains on data from the initial policy; online trains on data from the current policy. |
| 4 | What is the distribution mismatch problem in offline DPO? | The preference data was generated by policy_old, but we train policy_theta which diverges from policy_old. The DPO gradients become less informative as the distributions diverge. |
| 5 | What is iterative DPO? | Multiple rounds: DPO -> generate on-policy data -> rank with RM -> DPO -> repeat. Each round addresses failure modes the previous round could not see. Diminishing returns after 3-5 iterations. |
| 6 | What is SPIN (Self-Play Fine-Tuning)? | Model competes against its previous version. DPO pairs: chosen = human response, rejected = previous model's response. Converges when model distribution matches human distribution. No RM needed. |
| 7 | What is reward model overoptimization? | Optimizing too hard against an RM causes proxy reward to increase but true quality to decrease. Gao et al.: with d = sqrt(KL), gold reward follows d(alpha - beta*d) for best-of-n and d(alpha - beta*log d) for RL. There is an optimal KL budget. Goodhart's Law. |
| 8 | How do you prevent overoptimization? | KL budget/early stopping, RM ensembles, iterative re-labeling, conservative beta/KL coefficient, or use rejection sampling instead of gradient-based RL. |
| 9 | What is the Llama 3 post-training recipe? | Iterative rounds (~6) of: reward modeling on human preference data -> rejection sampling to build SFT data -> SFT on curated + rejection-sampled data -> DPO on human-annotated preference pairs. Simple methods at scale > complex methods at small scale. |
| 10 | When should you use PPO vs DPO vs rejection sampling? | PPO: maximum performance, large teams. DPO: good default, simple, stable. Rejection sampling + DPO: industrial scale, robust, parallelizable. |
| 11 | What is test-time compute scaling? | Spending more compute at inference (e.g., chain-of-thought, tree search, self-verification) to improve output quality. The new frontier beyond training-time scaling. |
| 12 | What determines the ceiling for post-training? | The base model's capabilities. Post-training can elicit and align existing capabilities, but cannot add fundamentally new ones. Invest in the best base model you can. |

## 11. Paper Reading Guides

### Paper 1: Iterative Preference Learning (Xiong et al. 2024)
- **Link**: https://arxiv.org/abs/2312.11456
- **Read first**: Section 3 -- the online iterative DPO framework
- **Key result**: Online DPO significantly outperforms offline DPO
- **Critical insight**: The distribution mismatch in offline DPO is not just theoretical -- it causes measurable degradation in practice, especially as training progresses
- **Interview angle**: "Online DPO is to offline DPO what on-policy RL is to off-policy RL -- same fundamental distinction, same reasons for the performance gap."

### Paper 2: SPIN (Chen et al. 2024)
- **Link**: https://arxiv.org/abs/2401.01335
- **Read first**: Section 2 -- the self-play formulation
- **Key figure**: Figure 2 -- convergence over iterations
- **Key result**: SPIN achieves competitive performance with DPO without needing a reward model or preference data
- **Critical insight**: The fixed point of SPIN is when the model's distribution matches the target (human) distribution. At convergence, model responses are indistinguishable from human responses, so there is nothing left to learn.
- **Interview angle**: "SPIN reframes alignment as a minimax game. The Nash equilibrium is when the model IS the human distribution."

### Paper 3: Scaling Laws for Reward Model Overoptimization (Gao et al. 2023)
- **Link**: https://arxiv.org/abs/2210.10760
- **Read first**: Section 3 -- the scaling law formulation
- **Key figure**: Figure 1 -- gold reward vs proxy reward as a function of optimization
- **Key result**: With d = sqrt(KL), gold reward follows d(alpha_bon - beta_bon*d) for best-of-n and d(alpha_RL - beta_RL*log d) for RL; the alpha/beta ratio (set by RM quality) determines the optimal KL budget.
- **Critical insight**: Better RMs (higher d/c ratio) allow more optimization. RM quality is the bottleneck.
- **Interview angle**: "Goodhart's Law is not just a pithy saying -- Gao et al. quantified it with a precise scaling law."

### Paper 4: Llama 3 (Meta, 2024)
- **Link**: https://arxiv.org/abs/2407.21783
- **Read first**: Section 4 -- Post-Training
- **Key insight**: Meta chose rejection sampling + DPO over PPO. At their scale, simplicity and parallelism matter more than algorithmic sophistication.
- **Key lesson**: The recipe is embarrassingly simple: iterated rounds of SFT (on curated + rejection-sampled data) and DPO (on human-annotated preference pairs). The engineering challenge is data quality and scale, not algorithmic novelty.
- **Interview angle**: "The Llama 3 recipe shows that at scale, the bottleneck is data quality and engineering, not algorithm choice. DPO + rejection sampling + iteration is enough."